# Titanic Passenger Survival Prediction System

**AIC354 Machine Learning Fundamentals — Assignment 2**

### LIVE DEPLOYMENT LINK
**Add your Streamlit live link here after deployment.**

This notebook follows the instructor's Recommended Titanic workflow:
Import Libraries → Load Data → Preprocess → Encode → Train → Test → Application → Feedback → Deploy.


## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from prettytable import PrettyTable
import matplotlib.pyplot as plt
import seaborn as sns


## Step 2: Load Sample Data

In [ ]:
sample_data = pd.read_csv("sample-data.csv")
pd.set_option("display.max_rows", None, "display.max_columns", None)
print(sample_data)


## Step 3: Understand and Pre-process Sample Data

The supplied 100-row sample dataset is already preprocessed, so no additional preprocessing is required.


In [ ]:
print("Attributes in Sample Data:")
print(sample_data.columns)
print("Number of Instances:", sample_data["PClass"].count())


## Step 4: Feature Extraction
Features are already extracted: PClass, Gender, Sibling, and Embarked.

## Step 5: Label Encoding

In [ ]:
pclass = pd.DataFrame({"Pclass": ["First", "Second", "Third"]})
gender = pd.DataFrame({"Gender": ["Male", "Female"]})
sibling = pd.DataFrame({"Sibling": ["Zero", "One", "Two", "Three"]})
embarked = pd.DataFrame({"Embarked": ["Southampton", "Cherbourg", "Queenstown"]})
survived = pd.DataFrame({"Survived": ["Yes", "No"]})

pclass_label_encoder = LabelEncoder().fit(np.ravel(pclass))
gender_label_encoder = LabelEncoder().fit(np.ravel(gender))
sibling_label_encoder = LabelEncoder().fit(np.ravel(sibling))
embarked_label_encoder = LabelEncoder().fit(np.ravel(embarked))
survived_label_encoder = LabelEncoder().fit(np.ravel(survived))


### Step 5.1: Label Encode the Output

In [ ]:
sample_data_encoded_output = sample_data.copy()
sample_data_encoded_output["Survived"] = survived_label_encoder.transform(sample_data["Survived"])
sample_data_encoded_output.to_csv("sample-data-encoded-output.csv", index=False)
print(sample_data_encoded_output[["Survived"]].head())


### Step 5.2: Label Encode the Input

In [ ]:
sample_data_encoded = sample_data.copy()
sample_data_encoded["PClass"] = pclass_label_encoder.transform(sample_data["PClass"])
sample_data_encoded["Gender"] = gender_label_encoder.transform(sample_data["Gender"])
sample_data_encoded["Sibling"] = sibling_label_encoder.transform(sample_data["Sibling"])
sample_data_encoded["Embarked"] = embarked_label_encoder.transform(sample_data["Embarked"])
sample_data_encoded["Survived"] = survived_label_encoder.transform(sample_data["Survived"])
sample_data_encoded.to_csv("sample-data-encoded.csv", index=False)
print(sample_data_encoded.head())


## Step 6: Execute the Training Phase

In [ ]:
training_data_encoded, testing_data_encoded = train_test_split(
    sample_data_encoded,
    test_size=0.2,
    random_state=0,
    shuffle=False
)
training_data_encoded.to_csv("training-data-encoded.csv", index=False)
testing_data_encoded.to_csv("testing-data-encoded.csv", index=False)
print("Training Data shape:", training_data_encoded.shape)
print("Testing Data shape:", testing_data_encoded.shape)


### Step 6.1: Split Input Vectors and Outputs

In [ ]:
input_vector_train = training_data_encoded.iloc[:, :-1]
output_label_train = training_data_encoded.iloc[:, -1]
print("Input vectors (train) shape:", input_vector_train.shape)
print("Output labels (train) shape:", output_label_train.shape)


### Step 6.2: Train the Support Vector Classifier

In [ ]:
svc_model = svm.SVC(gamma="auto", random_state=0)
svc_model.fit(input_vector_train, np.ravel(output_label_train))
print(svc_model)


### Step 6.3: Save the Trained Model

In [ ]:
pickle.dump(svc_model, open("svc_trained_model.pkl", "wb"))
print("Saved: svc_trained_model.pkl")


## Step 7: Execute the Testing Phase

In [ ]:
input_vector_test = testing_data_encoded.iloc[:, :-1]
output_label_test = testing_data_encoded.iloc[:, -1]
model = pickle.load(open("svc_trained_model.pkl", "rb"))
model_predictions = model.predict(input_vector_test)
testing_data_encoded["Predictions"] = model_predictions
testing_data_encoded.to_csv("model-predictions.csv", index=False)

model_accuracy_score = accuracy_score(
    testing_data_encoded["Survived"],
    testing_data_encoded["Predictions"]
)
print("Accuracy Score:", model_accuracy_score)
print("\nClassification Report:")
print(classification_report(
    testing_data_encoded["Survived"],
    testing_data_encoded["Predictions"],
    target_names=["Not Survived (0)", "Survived (1)"]
))
print("\nConfusion Matrix:")
print(confusion_matrix(
    testing_data_encoded["Survived"],
    testing_data_encoded["Predictions"]
))


## Step 8: Execute the Application Phase

In [ ]:
pclass_input = "First"
gender_input = "Male"
sibling_input = "Zero"
embarked_input = "Queenstown"

user_input = pd.DataFrame({
    "PClass": [pclass_input],
    "Gender": [gender_input],
    "Sibling": [sibling_input],
    "Embarked": [embarked_input]
})

unseen_data_features = user_input.copy()
unseen_data_features["PClass"] = pclass_label_encoder.transform(user_input["PClass"])
unseen_data_features["Gender"] = gender_label_encoder.transform(user_input["Gender"])
unseen_data_features["Sibling"] = sibling_label_encoder.transform(user_input["Sibling"])
unseen_data_features["Embarked"] = embarked_label_encoder.transform(user_input["Embarked"])

predicted_survival = model.predict(unseen_data_features)[0]
print("Prediction:", "SURVIVED" if predicted_survival == 1 else "NOT SURVIVED")


## Step 9: Execute the Feedback Phase

In [ ]:
feedback = pd.DataFrame({
    "Reviewer": ["Student/Domain Reviewer"],
    "Case": ["Titanic passenger prediction"],
    "Feedback": ["The model is simple and suitable for teaching; adding more real-world Titanic data could improve generalization."]
})
print(feedback)

improvements = [
    "Use a larger Titanic dataset",
    "Compare additional classification algorithms",
    "Tune SVC hyperparameters",
    "Collect more real-world feedback"
]
for i, item in enumerate(improvements, 1):
    print(f"{i}. {item}")


## Step 10: Deploy the Model as a Public Web App

The deployment files are included with this submission:
- `app.py`
- `svc_trained_model.pkl`
- `requirements.txt`

Upload these files to GitHub and deploy `app.py` through Streamlit Community Cloud.

**After deployment, replace the LIVE DEPLOYMENT LINK at the top of this notebook with the actual public URL.**
